### =============================================================================
### A Network-Sensitive Framework for Assessing the Feasibility of Distributed
### VQAs in Multi-QPU Architectures
### Script: Benchmark Circuit Reader and Dimension Extractor
### Author: Waldemir Cambiucci
### Date: March 16, 2026
####
### Description:
### This script reads benchmark quantum circuits for QAOA and VQE and extracts
### structural dimensions and circuit-level metrics required for feasibility
### analysis in distributed multi-QPU settings.
###
### Main tasks:
### - Load benchmark circuits
### - Parse circuit structure
### - Extract key dimensions and metrics
### - Support downstream analysis for distributed VQA feasibility assessment
### Inputs:
### - Benchmark circuit files (e.g., OpenQASM 2.0 / 3.0)
### Outputs:
### - Structured dataset with extracted circuit dimensions and metrics
###
### Project:
### Distributed VQA feasibility analysis under network-sensitive conditions
### =============================================================================

In [45]:
from pathlib import Path
from qiskit import qasm3

In [57]:
INPUT_LIST = "list_4_8_12_qubits.txt"
OUTPUT_FILE = "output.txt"

def safe_metric(func, default="ERROR"):
    """Run a metric function safely."""
    try:
        return func()
    except Exception:
        return default

In [58]:
def analyze_qasm_file(qasm_path: str) -> dict:
    """Load one QASM file and extract circuit metrics."""
    result = {
        "file": qasm_path,
        "status": "OK",
        "depth": "",
        "size": "",
        "width": "",
        "num_qubits": "",
        "num_clbits": "",
        "num_cx": 0,
        "num_cnot": 0,
        "num_rzz": 0,
        "num_rxx": 0,
        "num_cz": 0,
        "total_ops": "",
        "ops_breakdown": "",
        "error": "",
    }

    try:
        # 1) tentativa normal
        try:
            circuit = qasm2.load(qasm_path)
        except Exception:
            # 2) fallback para QASM2 legado, sem duplicar gates já suportadas
            circuit = qasm2.load(
                qasm_path,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                strict=False
            )

        ops = circuit.count_ops()
        cx_count = int(ops.get("cx", 0))
        rzz_count = int(ops.get("rzz", 0))
        rxx_count = int(ops.get("rxx", 0))
        cz_count = int(ops.get("cz", 0))

        result["depth"] = safe_metric(circuit.depth)
        result["size"] = safe_metric(circuit.size)
        result["width"] = safe_metric(circuit.width)
        result["num_qubits"] = circuit.num_qubits
        result["num_clbits"] = circuit.num_clbits
        result["num_cx"] = cx_count
        result["num_cnot"] = cx_count  # alias
        result["num_rzz"] = rzz_count
        result["num_rxx"] = rxx_count
        result["num_cz"] = cz_count
        result["total_ops"] = sum(int(v) for v in ops.values())
        result["ops_breakdown"] = ", ".join(f"{k}:{v}" for k, v in sorted(ops.items()))

    except Exception as e:
        result["status"] = "ERROR"
        result["error"] = str(e)

    return result

In [59]:
def analyze_qasm_file_old_ok(qasm_path: str) -> dict:
    """Load one QASM file and extract circuit metrics."""
    result = {
        "file": qasm_path,
        "status": "OK",
        "depth": "",
        "size": "",
        "width": "",
        "num_qubits": "",
        "num_clbits": "",
        "num_cx": 0,
        "num_cnot": 0,
        "total_ops": "",
        "ops_breakdown": "",
        "error": "",
    }

    try:
        # 1) tentativa normal
        try:
            circuit = qasm2.load(qasm_path)
        except Exception:
            # 2) fallback para QASM2 legado, sem duplicar gates já suportadas
            circuit = qasm2.load(
                qasm_path,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                strict=False
            )

        ops = circuit.count_ops()
        cx_count = int(ops.get("cx", 0))

        result["depth"] = safe_metric(circuit.depth)
        result["size"] = safe_metric(circuit.size)
        result["width"] = safe_metric(circuit.width)
        result["num_qubits"] = circuit.num_qubits
        result["num_clbits"] = circuit.num_clbits
        result["num_cx"] = cx_count
        result["num_cnot"] = cx_count  # alias
        result["total_ops"] = sum(int(v) for v in ops.values())
        result["ops_breakdown"] = ", ".join(f"{k}:{v}" for k, v in sorted(ops.items()))

    except Exception as e:
        result["status"] = "ERROR"
        result["error"] = str(e)

    return result

In [60]:
def read_paths_list(filename: str) -> list[str]:
    """Read non-empty lines from paths.txt."""
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {filename}")

    lines = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                lines.append(line)
    return lines

In [61]:
def write_results(results: list[dict], output_filename: str) -> None:
    """Write all results to output.txt."""
    from datetime import datetime

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename_with_ts = f"output_{timestamp}.txt"

    with open(output_filename_with_ts, "w", encoding="utf-8") as f:
        f.write("QISKIT CIRCUIT METRICS REPORT\n")
        f.write("=" * 100 + "\n\n")

        for i, r in enumerate(results, start=1):
            f.write(f"[{i}]\n")
            f.write(f"FILE: {r['file']}\n")
            f.write(f"STATUS: {r['status']}\n")

            if r["status"] == "OK":
                f.write(f"DEPTH: {r['depth']}\n")
                f.write(f"SIZE: {r['size']}\n")
                f.write(f"WIDTH: {r['width']}\n")
                f.write(f"NUMBER OF QUBITS: {r['num_qubits']}\n")
                f.write(f"NUMBER OF CLASSICAL BITS: {r['num_clbits']}\n")

                f.write(f"NUMBER OF CX / CNOT GATES: {r['num_cx']}\n")
                f.write(f"NUMBER OF CZ GATES: {r['num_cz']}\n")
                f.write(f"NUMBER OF RXX GATES: {r['num_rxx']}\n")
                f.write(f"NUMBER OF RZZ GATES: {r['num_rzz']}\n")

                f.write(f"TOTAL OPERATIONS: {r['total_ops']}\n")
                f.write(f"OPERATIONS BREAKDOWN: {r['ops_breakdown']}\n")
            else:
                f.write(f"ERROR: {r['error']}\n")

            f.write("-" * 100 + "\n")

        # Summary
        ok_count = sum(1 for r in results if r["status"] == "OK")
        err_count = len(results) - ok_count

        f.write("\nSUMMARY\n")
        f.write("=" * 100 + "\n")
        f.write(f"TOTAL FILES: {len(results)}\n")
        f.write(f"SUCCESSFULLY PROCESSED: {ok_count}\n")
        f.write(f"ERRORS: {err_count}\n")

In [62]:
def main():
    qasm_files = read_paths_list(INPUT_LIST)
    results = []
    for qasm_file in qasm_files:
        results.append(analyze_qasm_file(qasm_file))
    write_results(results, OUTPUT_FILE)
    print(f"Done. Results written to: {OUTPUT_FILE}")

In [63]:
if __name__ == "__main__":
    main()

Done. Results written to: output.txt
